# PPO Training: EdgeTransformerNet + Curriculum

Fine-tunes the imitation-pretrained EdgeTransformerNet via PPO.

## Required Kaggle inputs

| Dataset slug | Contents | Required? |
|---|---|---|
| orbitswar-src | Full project source + enemy agents | Always |
| orbitswar-imitation | best_model.pt from imitation training | Always |
| orbitswar-rl | transformer_last.pt + transformer_best.pt + history + enemy_stats | Resume only |

## Curriculum — manual via START_TIER

| START_TIER | Enemy pool | Games per round |
|---|---|---|
| 0 | Tier1 only (5 easy bots) | 32 vs easy |
| 1 | Tier1 + Tier2 (10 bots) | 16 easy + 16 medium, alternating |
| 2 | All tiers (15 bots) | ~11 each, cycling |

**Terminal rewards:** WIN = +6.0 | LOSS = -8.0 | Break-even = 57.1% WR

## Downloads after each session
transformer_last.pt, transformer_best.pt, transformer_history.json, enemy_stats.json, ppo_training.log
-> zip as orbitswar-rl, re-upload, resume next session.

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 1 — Install orbit_wars environment
# ─────────────────────────────────────────────────────────
%pip install --upgrade "kaggle-environments>=1.28.0"

# Verify orbit_wars is available
from kaggle_environments import make
env = make('orbit_wars', configuration={'episodeSteps': 5})
print('orbit_wars OK')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 2b — Logging setup
# Appends to previous log if one exists in orbitswar-rl
# dataset — so the full PPO training history accumulates
# across Kaggle sessions.
# ─────────────────────────────────────────────────────────
import sys, time, shutil
from pathlib import Path

LOG_PATH = Path('/kaggle/working/ppo_training.log')

# Check for previous log from the RL dataset (resume accumulation)
_prev_log = Path('/kaggle/input/datasets/anubhav123412/orbitswar-rl/ppo_training.log')
if _prev_log.exists() and not LOG_PATH.exists():
    shutil.copy2(_prev_log, LOG_PATH)
    _log_mode = 'a'
else:
    _log_mode = 'w'


class _TeeLogger:
    def __init__(self, orig, path):
        self._orig = orig
        self._file = open(path, _log_mode, buffering=1, encoding='utf-8')
        ts = time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
        if _log_mode == 'a':
            self._file.write(f'\n\n# ══ Resumed session : {ts} ══\n\n')
        else:
            self._file.write('# OrbitsWar — PPO Training Log\n')
            self._file.write(f'# Started : {ts}\n\n')
        self._file.flush()

    def write(self, msg):
        # 1. Jupyter cell output
        self._orig.write(msg)
        try: self._orig.flush()
        except: pass
        # 2. stderr → live in Kaggle Logs tab during committed runs
        try:
            sys.__stderr__.write(msg)
            sys.__stderr__.flush()
        except: pass
        # 3. log file
        try:
            self._file.write(msg)
            self._file.flush()
        except: pass

    def flush(self):
        try: self._orig.flush()
        except: pass
        try: sys.__stderr__.flush()
        except: pass
        try: self._file.flush()
        except: pass

    def isatty(self):
        return getattr(self._orig, 'isatty', lambda: False)()

    def close(self):
        sys.stdout = self._orig
        try: self._file.flush(); self._file.close()
        except: pass


_tee = _TeeLogger(sys.stdout, LOG_PATH)
sys.stdout = _tee

if _log_mode == 'a':
    prev_sz = _prev_log.stat().st_size / 1024
    print(f'Previous log found ({prev_sz:.1f} KB) — appending to existing history.')
print(f'Logging to : {LOG_PATH}  (mode={_log_mode})')
print('(stdout → ppo_training.log + Kaggle Logs tab live)')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 2 — Check GPU
# ─────────────────────────────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ---------------------------------------------------------
# CELL 3 — Dataset paths
# ---------------------------------------------------------
from pathlib import Path

KAGGLE_USER       = "anubhav123412"
SRC_DATASET       = "orbitswar-src"        # source code (required)
IMITATION_DATASET = "orbitswar-imitation"  # best_model.pt (required)
RL_DATASET        = "orbitswar-rl"          # PPO checkpoints (resume only)

BASE = f"/kaggle/input/datasets/{KAGGLE_USER}"

SRC_DIR  = Path(f"{BASE}/{SRC_DATASET}")
IMIT_DIR = Path(f"{BASE}/{IMITATION_DATASET}")

print(f"SRC  exists: {SRC_DIR.exists()}   ({SRC_DIR})")
print(f"IMIT exists: {IMIT_DIR.exists()}  ({IMIT_DIR})")

# RL dataset is only needed to RESUME a previous session.
# On a fresh start, leave it unattached -- RL_DIR will be None.
_rl_candidate = Path(f"{BASE}/{RL_DATASET}")
RL_DIR = _rl_candidate if _rl_candidate.exists() else None

if RL_DIR:
    print(f"RL   found : {RL_DIR}  <- will resume from previous session")
else:
    print("RL   not attached -- fresh start (no prior PPO checkpoints needed)")


In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 4 — Copy source tree to /kaggle/working/
# ─────────────────────────────────────────────────────────
import os, sys, shutil

WORK = Path('/kaggle/working')

# Copy everything from the source dataset
for item in SRC_DIR.iterdir():
    dst = WORK / item.name
    if item.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(item, dst)
    else:
        shutil.copy2(item, dst)
print('Source tree copied.')

# ── Copy imitation checkpoint ─────────────────────────────
imit_ckpt_dst = WORK / 'imitation' / 'checkpoints'
imit_ckpt_dst.mkdir(parents=True, exist_ok=True)
imit_src = IMIT_DIR / 'best_model.pt'
if imit_src.exists():
    shutil.copy2(imit_src, imit_ckpt_dst / 'best_model.pt')
    print(f'Imitation checkpoint copied -> {imit_ckpt_dst}/best_model.pt')
else:
    print('WARNING: best_model.pt not found — actor will start from random init.')

# ── Copy transformer checkpoints + history + log (resume only) ───────────────
rl_dst = WORK / 'rl'
rl_dst.mkdir(parents=True, exist_ok=True)
if RL_DIR is not None:
    resumed = []
    for name in ['transformer_last.pt', 'transformer_best.pt',
                 'transformer_history.json', 'ppo_training.log']:
        src = RL_DIR / name
        if src.exists():
            # log goes to working root (where _TeeLogger expects it)
            dst = WORK / name if name == 'ppo_training.log' else rl_dst / name
            shutil.copy2(src, dst)
            resumed.append(name)
    if resumed:
        print(f'PPO files copied: {resumed}')
        print('  trainer will resume automatically from transformer_last.pt')
        if 'transformer_history.json' in resumed:
            import json as _j
            _h = _j.loads((rl_dst / 'transformer_history.json').read_text())
            print(f'  History loaded: {len(_h)} previous rounds — curves will accumulate')
        if 'ppo_training.log' in resumed:
            log_kb = (WORK / 'ppo_training.log').stat().st_size / 1024
            print(f'  Log loaded: {log_kb:.1f} KB — will append to existing log')
    else:
        print('RL dataset attached but no files found — starting fresh.')
else:
    print('Fresh start — no prior PPO checkpoints.')

# ── Ensure enemy agents directory exists ──────────────────
enemy_dir = WORK / 'enemy agents'
if not enemy_dir.exists():
    enemy_dir.mkdir(parents=True)
std_bots = list(enemy_dir.glob('*/main.py'))
my_bots  = list((enemy_dir / 'my').glob('*.py')) if (enemy_dir / 'my').exists() else []
if not std_bots and not my_bots:
    self_bot_dir = enemy_dir / 'self_main'
    self_bot_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(WORK / 'main.py', self_bot_dir / 'main.py')
    print('No enemies found — added main.py as self-play opponent.')
else:
    labels = [p.parent.name for p in std_bots] + [p.stem for p in my_bots]
    print(f'Enemies: {labels}')

# ── Python path ───────────────────────────────────────────
for p in [str(WORK), str(WORK/'rl'), str(WORK/'imitation'), str(WORK/'classifier')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('\nWorking directory contents:')
for p in sorted(WORK.iterdir()):
    print(f'  {p.name}/' if p.is_dir() else f'  {p.name}')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 5 — Verify imports
# ─────────────────────────────────────────────────────────
os.chdir(WORK)   # make relative imports work from project root

from imitation.model    import EdgeTransformerNet, load_model as load_actor
from imitation.features import build_all_candidate_edges

# Count actor parameters
actor_test = EdgeTransformerNet()
n_params = sum(p.numel() for p in actor_test.parameters())
print(f'EdgeTransformerNet params: {n_params:,}')
del actor_test

# Check imitation checkpoint
imit_ckpt = WORK / 'imitation' / 'checkpoints' / 'best_model.pt'
print(f'Imitation checkpoint exists: {imit_ckpt.exists()}')
if imit_ckpt.exists():
    import torch
    ckpt = torch.load(imit_ckpt, map_location='cpu', weights_only=False)
    print(f'  Trained epoch: {ckpt.get("epoch", "?")}  '
          f'val_loss: {ckpt.get("val_loss", float("nan")):.4f}')

print('\nAll imports OK.')

In [ ]:
# ---------------------------------------------------------
# CELL 6 — Session configuration
# <- CHANGE THESE each session
# ---------------------------------------------------------
import rl.transformer_trainer as TT

N_ROUNDS            = 999      # run until Kaggle 12h limit
START_TIER          = 1        # 0=easy  1=medium  2=hard
START_TIER_PURE     = True     # True=only that tier  False=cumulative mix
RESET_ENT_ON_RESUME = True     # True: fresh entropy decay

_tier_names = ["Tier1 only (easy)", "Tier2 only (medium)", "Tier3 only (hard)"]
_mode       = "pure" if START_TIER_PURE else "cumulative"
print(f"N_ROUNDS            : {N_ROUNDS}")
print(f"START_TIER          : {START_TIER}  ({_tier_names[START_TIER]})  mode={_mode}")
print(f"RESET_ENT_ON_RESUME : {RESET_ENT_ON_RESUME}")
print()
print("Hyperparameters (from transformer_trainer.py):")
print(f"  LR_ACTOR   : {TT.LR_ACTOR}")
print(f"  LR_CRITIC  : {TT.LR_CRITIC}")
print(f"  CLIP_EPS   : {TT.CLIP_EPS}")
print(f"  PPO_EPOCHS : {TT.PPO_EPOCHS}")
print(f"  TARGET_KL  : {TT.TARGET_KL}")
print(f"  ENT_COEF   : {TT.ENT_COEF_START} -> {TT.ENT_COEF_END} over {TT.ENT_COEF_RAMP} rounds")
print(f"  TERMINAL   : win={TT.TERMINAL_WIN}  loss={TT.TERMINAL_LOSS}  break-even={abs(TT.TERMINAL_LOSS)/(TT.TERMINAL_WIN+abs(TT.TERMINAL_LOSS)):.1%}")
print(f"  EVENT_SCALE: {TT._E}")
print(f"  WARMUP     : {TT.CRITIC_WARMUP_ROUNDS} rounds (skipped on resume)")
print()
print("Curriculum tiers:")
for i, tier in enumerate(TT.CURRICULUM_TIERS_1V1):
    print(f"  Tier {i+1}: {tier}")


In [ ]:
# ---------------------------------------------------------
# CELL 7 — Launch training
# ---------------------------------------------------------
import rl.transformer_trainer as TT
import torch

TT.N_ROUNDS            = N_ROUNDS
TT.GAMES_PER_ROUND     = 32
TT.START_TIER          = START_TIER
TT.START_TIER_PURE     = START_TIER_PURE
TT.RESET_ENT_ON_RESUME = RESET_ENT_ON_RESUME

TT.SAVE_PATH         = WORK / "rl" / "transformer_last.pt"
TT.BEST_PATH         = WORK / "rl" / "transformer_best.pt"
TT.HISTORY_PATH      = WORK / "rl" / "transformer_history.json"
TT.CURVES_PATH       = WORK / "rl" / "transformer_curves.png"
TT.ENEMY_STATS_PATH  = WORK / "rl" / "enemy_stats.json"

for _ckpt_path in [TT.SAVE_PATH, TT.BEST_PATH]:
    if _ckpt_path.exists():
        _ckpt = torch.load(_ckpt_path, map_location="cpu", weights_only=False)
        _ckpt["best_wr"] = 0.0
        _ckpt["best_wr_ppo"] = 0.0
        torch.save(_ckpt, _ckpt_path)
        print(f"Reset best_wr_ppo -> 0.0 in {_ckpt_path.name}")

print(f"Checkpoints -> last : {TT.SAVE_PATH}")
print(f"              best : {TT.BEST_PATH}")
print(f"History            : {TT.HISTORY_PATH}")
print(f"Enemy stats        : {TT.ENEMY_STATS_PATH}")
print()

actor = critic = None
try:
    actor, critic = TT.train(
        n_rounds        = N_ROUNDS,
        games_per_round = 32,
        actor_path      = str(WORK / "imitation" / "checkpoints" / "best_model.pt"),
        device          = DEVICE,
    )
except KeyboardInterrupt:
    print("Training interrupted.")
finally:
    from IPython.display import Image, display
    if TT.CURVES_PATH.exists():
        display(Image(str(TT.CURVES_PATH)))


In [ ]:
# ---------------------------------------------------------
# CELL 8 — Quick sanity evaluation (1v1 vs easy + medium + hard)
# ---------------------------------------------------------
from rl.transformer_trainer import play_game_collect
from rl.self_play_trainer import load_agent

if actor is None:
    print("Training returned None -- skipping eval.")
else:
    eval_opponents = [
        ("tamrazov-starwars",          "easy  ", 0),
        ("sigmaborov-starter",         "medium", 1),
        ("ykhnkf-distance-prioritized","hard  ", 2),
    ]

    print("Quick eval -- 4 x 1v1 per opponent:")
    print(f'  {"Opponent":<32} {"W":>2} {"L":>2}  {"WR":>6}')
    print("  " + "-" * 46)

    for label, tier_name, tier_idx in eval_opponents:
        agent_path = WORK / "enemy agents" / label / "main.py"
        if not agent_path.exists():
            print(f"  {label:<32} not found, skipping")
            continue
        try:
            opp_fn, _ = load_agent(str(agent_path))
        except Exception as e:
            print(f"  {label:<32} load error: {e}")
            continue

        w = l = 0
        for g in range(4):
            buf, outcome = play_game_collect(
                actor, critic, [None, opp_fn], seed=8000 + g,
                device=DEVICE)
            if outcome == "WIN": w += 1
            else:                l += 1
        wr = w / 4
        print(f"  [{tier_name}] {label:<28} {w:>2} {l:>2}  {wr:>5.0%}")

    print()
    print("Expected if learning: easy ~75%+, medium ~35-50%, hard ~15-30%")


In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 8b — Training curves + summary
#            Run any time to see current progress
# ─────────────────────────────────────────────────────────
import json
from IPython.display import Image, display

hist_path   = WORK / 'rl' / 'transformer_history.json'
curves_path = WORK / 'rl' / 'transformer_curves.png'

if not hist_path.exists():
    print('No history file yet — run training first.')
else:
    history = json.loads(hist_path.read_text())

    if not history:
        print('History file is empty.')
    else:
        # Show the saved PNG (generated by trainer after every round)
        if curves_path.exists():
            display(Image(str(curves_path)))
        else:
            print('No curves PNG yet — will appear after first round completes.')

        # Summary table
        best  = max(history, key=lambda h: h['wr'])
        last  = history[-1]
        warmup_rounds = [h for h in history if h.get('freeze_actor', False)]

        print(f'\n{"─"*55}')
        print(f'  Total rounds : {len(history)}')
        print(f'  Warmup rounds: {len(warmup_rounds)}  (actor frozen)')
        print(f'  Best WR      : {best["wr"]:.1%}  @ round {best["round"]}')
        print(f'{"─"*55}')
        print(f'  Last round ({last["round"]}):')
        print(f'    WR        = {last["wr"]:.1%}  '
              f'({last["wins"]}W {last["losses"]}L)')
        print(f'    loss      = {last["loss"]:.4f}')
        print(f'    entropy   = {last["entropy"]:.4f}')
        print(f'    KL        = {last["kl"]:.4f}')
        print(f'    clipfrac  = {last["clipfrac"]:.3f}')
        print(f'    ent_coef  = {last["ent_coef"]:.4f}')
        print(f'  Reward breakdown:')
        print(f'    events    = {last["events"]:+.2f}')
        print(f'    momentum  = {last["momentum"]:+.2f}')
        print(f'    terminal  = {last["terminal"]:+.2f}')
        print(f'    total     = {last["total_reward"]:+.2f}')
        print(f'{"─"*55}')

        # Quick reward health check
        if last['terminal'] > 0 and last['events'] < 0:
            print('  ⚠ Warning: terminal positive but events negative')
            print('    Agent may be winning by time, not planet control')
        if abs(last['momentum']) > abs(last['events']) * 2:
            print('  ⚠ Warning: momentum >> events — check for reward hacking')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 9 — List output files (download from Output tab)
# ─────────────────────────────────────────────────────────
import os

print('Output files (download ALL from Output tab):')
print()
for name in ['transformer_last.pt', 'transformer_best.pt',
             'transformer_history.json', 'enemy_stats.json', 'transformer_curves.png']:
    f = WORK / 'rl' / name
    if f.exists():
        sz = f.stat().st_size / 1e6
        print(f'  rl/{name:<35}  {sz:.1f} MB')
    else:
        print(f'  rl/{name:<35}  (not found)')

log = Path('/kaggle/working/ppo_training.log')
if log.exists():
    print(f'  ppo_training.log{"":<27}  {log.stat().st_size/1e6:.2f} MB')

print()
print('Resume workflow:')
print('  1. Download all files listed above')
print('  2. Place locally:')
print('       transformer_last.pt      → rl/transformer_last.pt')
print('       transformer_best.pt      → rl/transformer_best.pt')
print('       transformer_history.json → rl/transformer_history.json')
print('       ppo_training.log         → rl/ppo_training.log')
print('  3. python zip_kaggle_dataset.py --models')
print('     → orbitswar-rl.zip contains all 4 files')
print('  4. Upload orbitswar-rl.zip to Kaggle, rerun notebook')
print('     → weights resume, curves accumulate, log appends')

# ── Flush & close log ─────────────────────────────────────
try:
    import time as _t
    print(f'\n# Log ended : {_t.strftime("%Y-%m-%d %H:%M:%S UTC", _t.gmtime())}')
    _tee.flush()
    _log_sz = LOG_PATH.stat().st_size
    _tee.close()
    print(f'\nLog saved   : {LOG_PATH}  ({_log_sz / 1024:.1f} KB)')
except Exception as _log_err:
    print(f'Log close error: {_log_err}')